# 05 · Scraping gambar HP dari GSMArena

Notebook ini mengambil **foto HP** dari halaman GSMArena untuk setiap model di `data/raw_dataset_gsmarena.csv`, supaya dashboard bisa menampilkan galeri HP (Tab Ringkasan).

**Hasil**
- `data/phone_images/<phone_id>.jpg` — semua gambar dibuat berukuran **sama (300 × 400 px)**
- `data/phone_images.csv` — daftar gambar + URL sumber di GSMArena (untuk lampiran laporan)

**Cara pakai:** jalankan sel dari atas ke bawah. Lakukan **uji coba 5 gambar** dulu, cek hasilnya di sel pratinjau, baru jalankan versi penuh.

**Catatan**
- `phone_id` dibuat dengan rumus yang sama seperti `sql/gsmarena/02_staging.sql`, jadi otomatis cocok dengan `dim_phone` di dashboard.
- Aman dihentikan lalu dijalankan ulang: gambar yang sudah ada dilewati (checkpoint).
- Jeda antar request jangan diperkecil, GSMArena membatasi laju request.

In [ ]:
%pip install -q requests beautifulsoup4 lxml pandas pillow

In [ ]:
import random
import re
import time
from io import BytesIO
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from PIL import Image

In [ ]:
# Cari folder root proyek otomatis (folder yang berisi data/raw_dataset_gsmarena.csv),
# jadi notebook ini jalan baik saat working directory = root repo maupun = folder notebook/.
ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "data" / "raw_dataset_gsmarena.csv").exists():
        ROOT = p
        break
else:
    raise FileNotFoundError("data/raw_dataset_gsmarena.csv tidak ditemukan. Buka notebook dari dalam folder repo.")

CSV_IN = ROOT / "data" / "raw_dataset_gsmarena.csv"
IMG_DIR = ROOT / "data" / "phone_images"
MANIFEST = ROOT / "data" / "phone_images.csv"
IMG_DIR.mkdir(parents=True, exist_ok=True)

CANVAS = (300, 400)      # lebar x tinggi, sama untuk semua HP
DELAY = (6.0, 12.0)      # jeda acak antar request (detik); jangan diperkecil, GSMArena membatasi laju request

print("Root proyek :", ROOT)
print("Folder gambar:", IMG_DIR)

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.gsmarena.com/",
})


def make_phone_id(phone_name):
    # Sama persis dengan SQL staging: trim('_' dari regexp_replace(replace(lower(nama), '+', ' plus'), '[^a-z0-9]+', '_'))
    s = phone_name.strip().lower().replace("+", " plus")
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_")


class DiblokirError(Exception):
    """GSMArena menolak request (429/403): hentikan scraping, jangan dipaksa."""


def get(url, retries=3):
    """Kembalikan (response, alasan_gagal). Berhenti total kalau diblokir (429/403)."""
    alasan = "tidak diketahui"
    for attempt in range(retries):
        try:
            r = SESSION.get(url, timeout=(10, 45))
            if r.status_code == 200:
                return r, None
            if r.status_code in (429, 403):
                raise DiblokirError(f"HTTP {r.status_code}")
            alasan = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            alasan = f"error koneksi ({type(e).__name__})"
        time.sleep(5 * (attempt + 1))
    return None, alasan


def find_image_url(page_url):
    r, alasan = get(page_url)
    if r is None:
        return None, alasan
    soup = BeautifulSoup(r.text, "lxml")
    img = soup.select_one("div.specs-photo-main img")
    if img is None or not img.get("src"):
        return None, "halaman terbuka tapi foto tidak ditemukan"
    return img.get("src"), None


def save_normalized(img_bytes, path):
    """Taruh gambar di kanvas putih berukuran tetap tanpa mengubah proporsinya."""
    img = Image.open(BytesIO(img_bytes)).convert("RGBA")
    bg = Image.new("RGBA", img.size, (255, 255, 255, 255))      # transparansi -> putih
    img = Image.alpha_composite(bg, img).convert("RGB")
    img.thumbnail(CANVAS, Image.LANCZOS)
    canvas = Image.new("RGB", CANVAS, (255, 255, 255))
    canvas.paste(img, ((CANVAS[0] - img.width) // 2, (CANVAS[1] - img.height) // 2))
    canvas.save(path, "JPEG", quality=88)

In [ ]:
df = pd.read_csv(CSV_IN, encoding="utf-8-sig")
df["phone_id"] = df["phone_name"].map(make_phone_id)
df = df.drop_duplicates("phone_id").reset_index(drop=True)

sudah = sum((IMG_DIR / f"{pid}.jpg").exists() for pid in df["phone_id"])
print(f"{len(df)} model di CSV • {sudah} gambar sudah ada • {len(df) - sudah} belum diambil")
df[["brand", "phone_name", "phone_id", "gsmarena_url"]].head()

In [ ]:
def scrape_images(limit=None):
    """Ambil gambar yang belum ada. limit = batas jumlah gambar BARU (None = semua)."""
    old = pd.read_csv(MANIFEST) if MANIFEST.exists() else pd.DataFrame()
    src_map = dict(zip(old.get("phone_id", []), old.get("image_source_url", [])))
    fetched = fail = 0

    try:
        for i, r in enumerate(df.itertuples(), 1):
            out = IMG_DIR / f"{r.phone_id}.jpg"
            if out.exists():                                   # checkpoint
                continue
            if limit is not None and fetched >= limit:
                break
            time.sleep(random.uniform(*DELAY))
            src, alasan = find_image_url(r.gsmarena_url)
            resp = None
            if src:
                resp, alasan = get(src)
            if resp is None:
                print(f"[{i}/{len(df)}] GAGAL  {r.phone_name}  -> {alasan}")
                fail += 1
                if fail >= 3:
                    print("Berhenti: 3 kegagalan berturut-turut. Cek alasan di atas.")
                    break
                continue
            fail = 0
            save_normalized(resp.content, out)
            src_map[r.phone_id] = src
            fetched += 1
            print(f"[{i}/{len(df)}] OK     {r.phone_name}")
    except DiblokirError as e:
        print(f"\n⛔ DIBLOKIR sementara oleh GSMArena ({e}). Scraping dihentikan supaya blokir tidak makin lama.")
        print("   Tunggu 30-60 menit ATAU ganti jaringan (misalnya hotspot HP), lalu jalankan ulang sel ini.")

    # daftar gambar dibuat dari SEMUA file yang ada, bukan hanya yang baru diambil
    rows = [(r.phone_id, r.brand, r.phone_name, r.gsmarena_url, f"{r.phone_id}.jpg", src_map.get(r.phone_id, ""))
            for r in df.itertuples() if (IMG_DIR / f"{r.phone_id}.jpg").exists()]
    pd.DataFrame(rows, columns=["phone_id", "brand", "phone_name", "gsmarena_url",
                                "image_file", "image_source_url"]).to_csv(MANIFEST, index=False, encoding="utf-8-sig")
    print(f"\nSelesai. Gambar baru: {fetched} • total tersedia: {len(rows)} dari {len(df)} model")

## Cek koneksi ke GSMArena (jalankan dulu kalau sebelumnya gagal terus)
Sel ini hanya membuka 1 halaman dan menampilkan status HTTP-nya.
- `200` → aman, lanjut scraping.
- `429` atau `403` → IP kamu sedang dibatasi. Tunggu 30–60 menit atau ganti jaringan (hotspot HP), lalu cek lagi.

In [ ]:
def cek_koneksi():
    pending = df[~df["phone_id"].map(lambda p: (IMG_DIR / f"{p}.jpg").exists())]
    if pending.empty:
        print("Semua gambar sudah ada.")
        return
    url = pending.iloc[0]["gsmarena_url"]
    try:
        r = SESSION.get(url, timeout=(10, 30))
        print("URL    :", url)
        print("Status :", r.status_code, "(200 = aman, 429/403 = dibatasi)")
        print("Foto ditemukan:", BeautifulSoup(r.text, "lxml").select_one("div.specs-photo-main img") is not None)
    except requests.RequestException as e:
        print("Gagal terhubung:", type(e).__name__)

cek_koneksi()

## Uji coba (5 gambar)
Jalankan ini dulu. Kalau semuanya `OK` dan gambar di sel pratinjau terlihat benar, lanjut ke versi penuh.

In [ ]:
scrape_images(limit=5)

In [ ]:
def pratinjau(n=12, kolom=6):
    """Tampilkan beberapa gambar hasil scraping dalam satu kolase."""
    files = sorted(IMG_DIR.glob("*.jpg"))[:n]
    if not files:
        print("Belum ada gambar.")
        return
    tampil = [Image.open(f) for f in files]
    w, h = CANVAS
    baris = (len(tampil) + kolom - 1) // kolom
    kolase = Image.new("RGB", (kolom * w, baris * h), (230, 230, 230))
    for k, im in enumerate(tampil):
        kolase.paste(im, ((k % kolom) * w, (k // kolom) * h))
    kolase.thumbnail((1500, 1500))
    display(kolase)
    print("File:", ", ".join(f.name for f in files))

pratinjau()

## Versi penuh (per batch)
GSMArena membatasi laju request, jadi ambil **per batch 40 gambar**: jalankan sel di bawah, tunggu selesai, **istirahat beberapa menit**, lalu jalankan lagi sampai jumlah "belum diambil" habis. Satu batch sekitar 5–8 menit.
Kalau muncul pesan ⛔ DIBLOKIR, berhenti dulu 30–60 menit atau ganti jaringan (hotspot HP). Gambar yang sudah ada tidak diambil ulang.

In [ ]:
scrape_images(limit=40)

## Pemeriksaan hasil

In [ ]:
files = sorted(IMG_DIR.glob("*.jpg"))
ukuran = {Image.open(f).size for f in files}
print(f"Jumlah gambar : {len(files)} dari {len(df)} model")
print(f"Ukuran gambar : {ukuran}  (harus satu ukuran saja: {CANVAS})")

hilang = df[~df["phone_id"].isin([f.stem for f in files])]
print(f"\nModel tanpa gambar: {len(hilang)}")
if len(hilang):
    display(hilang[["brand", "phone_name", "gsmarena_url"]].head(20))
pd.read_csv(MANIFEST).head()

## Langkah berikutnya
1. Commit folder `data/phone_images/` dan file `data/phone_images.csv` ke Git (ukurannya kecil), supaya anggota kelompok lain tidak perlu scraping ulang.
2. Jalankan dashboard: `python -m streamlit run dashboard/dashboard.py`. Galeri HP muncul di tab **Ringkasan**.
3. Model yang gambarnya gagal diambil tetap tampil di galeri dengan ikon 📱 sebagai pengganti.